In [1]:
import random

#Takes n and s as inputs and returns a list-- representing the resulting heap sizes-- where its entries sum to s and there are n entries

n=4
s=10

def random_placement(n, s):

    player = 1 if s % 2 == 0 else 2 #Determines who takes the first turn of Phase 2
    
    heaps = [0] * n #create list with n entries

    for i in range(s):

        heap = random.randint(0, n-1)  #randomly selects one of the heaps (entries)
        heaps[heap] += 1   #Adds value of 1 to that entry


    return heaps, player

heaps, starter = random_placement(n, s)
print("Final heaps:", heaps)
print("Player starting Phase 2:", starter)






Final heaps: [3, 3, 1, 3]
Player starting Phase 2: 1


In [2]:
#We want to repeat this experiment a large number of times

simulations = 10000

total_largest = 0 #Heap with the highest no. of stones
total_empty = 0 #How many empty heaps there are
all_occupied = 0 #How often Phase 1 finishes with no empty heaps

for i in range(simulations):

    heaps, starter = random_placement(n, s)

    total_largest += max(heaps) #sums all largest heaps
    total_empty += heaps.count(0) #sums all no. of empty heaps
    
    if 0 not in heaps:
        all_occupied += 1



print("Number of simulations:", simulations)
#averages of largest heaps and no. of empty heaps
print("Average largest heap:", total_largest / simulations)
print("Average empty heaps:", total_empty / simulations)
print("Proportion with all heaps occupied:", all_occupied / simulations)





Number of simulations: 10000
Average largest heap: 4.2025
Average empty heaps: 0.2243
Proportion with all heaps occupied: 0.7812


In [3]:

#calculating the nim_sum 

def nim_sum(heaps):

    total = 0    #start at 0 as [0 \oplus a = a]   -> starting at zero does not change the first heap's value

    for heap in heaps:
        total = total ^ heap    #calculates the XOR of the current total and the next heap

    return total


#testing  nim-sum against some manually calculated examples
test_heaps = [ [3, 2, 1], [3, 2, 4, 1], [1, 1, 1], [2, 2], [4, 4, 4] ]

for heaps in test_heaps:

    print(heaps, ":", nim_sum(heaps))




[3, 2, 1] : 0
[3, 2, 4, 1] : 4
[1, 1, 1] : 1
[2, 2] : 0
[4, 4, 4] : 4


In [4]:

#Defining a function for identifying winning positions

def is_winning(heaps):

    non_empty = [heap for heap in heaps if heap > 0]     #removes all empty heaps

    #Case A: all heaps contain at most one stone
    if all(heap == 1 for heap in non_empty):   #checking whether we're in case A

        return len(non_empty) % 2 == 0 #Returns true if there is an even number of non-empty heaps (winning position)
        
    #Case B: at least one heap contains more than one stone
    return nim_sum(heaps) != 0    #Otherwise, consider Case B: returns true if the Nim-sum is non-zero (winning position)



#TESTING

#where a 'false' return means a losing position and a 'true' returns means a winning position
test_positions = [ [1], [1, 1], [1, 1, 1], [1, 1, 1, 1], [2], [2, 1], [2, 2], [3, 2, 4, 1], [0, 1, 1], [0, 0, 2] ]

for heaps in test_positions:
    print(heaps, ":", is_winning(heaps))


[1] : False
[1, 1] : True
[1, 1, 1] : False
[1, 1, 1, 1] : True
[2] : True
[2, 1] : True
[2, 2] : False
[3, 2, 4, 1] : True
[0, 1, 1] : True
[0, 0, 2] : True


In [5]:

#checking the positions of every possible move

def is_winning_recursive(heaps):

    #no stones remain: the previous player lost and we've already won :D
    if sum(heaps) == 0:
        return True

    #trying every possible legal move
    for i in range(len(heaps)):   #consider each heap seperately
        for remove in range(1, heaps[i] + 1):   #consider every possible number of stones that can be removed from that heap ^

            new_heaps = heaps.copy()
            new_heaps[i] -= remove

            # If one move leaves a losing position, this is winning
            if not is_winning_recursive(new_heaps):    
                return True

    # No move leaves the opponent in a losing position
    return False


In [6]:

from itertools import product

#Compareing the Nim-Sum and Recursive methods 

disagreements = 0
total = 0

for n in range(1, 5):   #investigating configurations between 0 and 4 heaps

    for heaps in product(range(5), repeat=n): #generates every possible configuration for each value of n

        mathematical = is_winning(list(heaps))
        recursive = is_winning_recursive(list(heaps))

        total += 1      #counting configurations

        if mathematical != recursive:     #identifying dissagreements

            disagreements += 1
            print("Disagreement:", heaps)


print("Configurations tested:", total)
print("Disagreements:", disagreements)


Configurations tested: 780
Disagreements: 0


In [7]:
def choose_move(heaps):

    #trying every possible legal move - lifted from is_winning_recusrive function
    for i in range(len(heaps)):

        for remove in range(1, heaps[i] + 1):

            new_heaps = heaps.copy()
            new_heaps[i] -= remove

            # Check if the opponent would be in a losing position
            if not is_winning(new_heaps):

                return i, remove

    # If no winning move exists, choose any legal move
    for i in range(len(heaps)):

        if heaps[i] > 0:
            return i, 1

    return None


In [8]:

#Playing a complete game


def play_nim(initial_heaps, starter):    # initial_heaps: the starting configuration of heaps. Starter: Player that takes the first move.

    #Phase 1 determines the sizes of the heaps and which player starts Phase 2
    heaps = initial_heaps.copy()   #heaps will be changed throughout the game.
    player = starter
    turn = 1        #just a turn counter

    print("Starting heaps:", heaps)         #displays initial configuration
    print()

    while sum(heaps) > 0:       #sum of stones in the heaps: a checker to ensure the game isn't over

        #choose a move, with i the heap we're removing from and remove the number of stones we're removing
        i, remove = choose_move(heaps)

        #makes the move we just chose
        heaps[i] -= remove

        #displays the move we just made
        print("Turn:", turn)
        print("Player", player, "removes", remove,
              "from heap", i + 1)
        print("Remaining heaps:", heaps)
        print()

        #checks whether the last stone was removed
        if sum(heaps) == 0:

            winner = 3 - player

            print("Player", player, "took the final stone.")
            print("Player", winner, "wins!")

            return winner

        #if not hange player and take next round
        player = 3 - player
        turn += 1



#Testing it:

initial_heaps = [3, 2, 4, 1]

play_nim(initial_heaps, 1)        #We now provide who plays the first move of Phrase 1 (by default falling to Player 1, as said in assumptions)

Starting heaps: [3, 2, 4, 1]

Turn: 1
Player 1 removes 4 from heap 3
Remaining heaps: [3, 2, 0, 1]

Turn: 2
Player 2 removes 1 from heap 1
Remaining heaps: [2, 2, 0, 1]

Turn: 3
Player 1 removes 1 from heap 4
Remaining heaps: [2, 2, 0, 0]

Turn: 4
Player 2 removes 1 from heap 1
Remaining heaps: [1, 2, 0, 0]

Turn: 5
Player 1 removes 2 from heap 2
Remaining heaps: [1, 0, 0, 0]

Turn: 6
Player 2 removes 1 from heap 1
Remaining heaps: [0, 0, 0, 0]

Player 2 took the final stone.
Player 1 wins!


1

In [9]:

#Investigating the probability that

#the player starting Phase 2 wins given a randomly generated Phase 1
#Player 1 (the player starting Phase 1) wins given a randomly generated Phase 1



#player1_wins tells us which original player benefits from the entire game.

simulations = 10000

starter_wins = 0
player1_wins = 0

for i in range(simulations):

    heaps, starter = random_placement(n, s)

    if is_winning(heaps):

        winner = starter

    else:

        winner = 3 - starter

    if winner == starter:
        starter_wins += 1

    if winner == 1:
        player1_wins += 1


print("Probability of Phase 2 starter winning:",
      starter_wins / simulations)

print("Probability of Player 1 winning:",
      player1_wins / simulations)


Probability of Phase 2 starter winning: 0.8
Probability of Player 1 winning: 0.8


In [10]:

#investigating the properties of random placement over many simulations

simulations = 10000

total_empty = 0
total_largest = 0
all_occupied = 0

for i in range(simulations):

    heaps, starter = random_placement(n, s)

    empty = heaps.count(0)       #counts the number of empty heaps
    largest = max(heaps)         #finds the size of the largest heap

    total_empty += empty
    total_largest += largest

    if empty == 0:
        all_occupied += 1


#Calculating the averages and proportion of fully occupied games

average_empty = total_empty / simulations
average_largest = total_largest / simulations
proportion_occupied = all_occupied / simulations

#printing
print("Average number of empty heaps:", average_empty)
print("Average size of the largest heap:", average_largest)
print("Proportion of games where every heap is occupied:", proportion_occupied)
print()

#comparing the simulated average number of empty heaps with the theoretical expectation

expected_empty = n * (1 - 1/n)**s

print("Theoretical expected number of empty heaps:", expected_empty)
print("Difference between simulation and theory:", abs(average_empty - expected_empty))


Average number of empty heaps: 0.2287
Average size of the largest heap: 4.2009
Proportion of games where every heap is occupied: 0.7762

Theoretical expected number of empty heaps: 0.22525405883789062
Difference between simulation and theory: 0.0034459411621093616


In [11]:

from functools import lru_cache


#determining whether a player can force a victory through strategic Phase 1 placement

def phase1_winning(heaps, remaining):

    #once all stones have been placed, we can havePhase 2 begins
    if remaining == 0:

        #the player whose turn it is would start Phase 2
        return is_winning(heaps)


    #trying every possible heap in which to place the next stone
    checked = set()       #avoids considering heaps of identical sizes more than once

    for i in range(len(heaps)):

        if heaps[i] in checked:
            continue

        checked.add(heaps[i])

        new_heaps = list(heaps)
        new_heaps[i] += 1

        #sorts the heaps so equivalent configurations are treated as the same position
        new_heaps = tuple(sorted(new_heaps))

        #If the opponent cannot force a victory, this is a winning move by def
        if not phase1_winning(new_heaps, remaining - 1):

            return True


    #no placement guarantees a victory, still have to have optimal/strategic choices.
    return False


In [12]:

#choosing a strategic move during Phase 1

def choose_phase1_move(heaps, remaining):

    #Trying every possible heap in which to place the next stone
    checked = set()

    for i in range(len(heaps)):

        if heaps[i] in checked:
            continue

        checked.add(heaps[i])

        new_heaps = heaps.copy()
        new_heaps[i] += 1

        new_heaps = tuple(sorted(new_heaps))

        #Check whether this placement leaves the opponent in a losing position
        if not phase1_winning(new_heaps, remaining - 1):

            return i


    #If no winning placement exists, choose any heap
    return 0


In [13]:

#before testing any larger results we'll check it actually works

#Testing strategic Phase 1 placement

test_cases = [(1, 1), (1, 2), (2, 1), (2, 2), (3, 2)]

for n_test, s_test in test_cases:

    initial_heaps = tuple([0] * n_test)

    result = phase1_winning(initial_heaps, s_test)

    if result:
        print("n =", n_test, ", s =", s_test, ": Player 1 can force a victory")
    else:
        print("n =", n_test, ", s =", s_test, ": Player 2 can force a victory")


n = 1 , s = 1 : Player 1 can force a victory
n = 1 , s = 2 : Player 1 can force a victory
n = 2 , s = 1 : Player 1 can force a victory
n = 2 , s = 2 : Player 1 can force a victory
n = 3 , s = 2 : Player 1 can force a victory


In [14]:

#Investigating strategic Phase 1 outcomes for different values of n and s

for n_test in range(1, 5):

    print("\nNumber of heaps:", n_test)

    for s_test in range(1, 13):

        initial_heaps = tuple([0] * n_test)

        result = phase1_winning(initial_heaps, s_test)

        if result:
            winner = "Player 1"
        else:
            winner = "Player 2"

        print("Stones:", s_test, "->", winner)



Number of heaps: 1
Stones: 1 -> Player 1
Stones: 2 -> Player 1
Stones: 3 -> Player 2
Stones: 4 -> Player 1
Stones: 5 -> Player 2
Stones: 6 -> Player 1
Stones: 7 -> Player 2
Stones: 8 -> Player 1
Stones: 9 -> Player 2
Stones: 10 -> Player 1
Stones: 11 -> Player 2
Stones: 12 -> Player 1

Number of heaps: 2
Stones: 1 -> Player 1
Stones: 2 -> Player 1
Stones: 3 -> Player 2
Stones: 4 -> Player 2
Stones: 5 -> Player 2
Stones: 6 -> Player 2
Stones: 7 -> Player 2
Stones: 8 -> Player 2
Stones: 9 -> Player 2
Stones: 10 -> Player 2
Stones: 11 -> Player 2
Stones: 12 -> Player 2

Number of heaps: 3
Stones: 1 -> Player 1
Stones: 2 -> Player 1
Stones: 3 -> Player 2
Stones: 4 -> Player 1
Stones: 5 -> Player 2
Stones: 6 -> Player 2
Stones: 7 -> Player 2
Stones: 8 -> Player 1
Stones: 9 -> Player 2
Stones: 10 -> Player 1
Stones: 11 -> Player 2
Stones: 12 -> Player 1

Number of heaps: 4
Stones: 1 -> Player 1
Stones: 2 -> Player 1
Stones: 3 -> Player 2
Stones: 4 -> Player 1
Stones: 5 -> Player 2
Stones: 6